### Registration of instruments and chemicals in division (FB) and BAM inventory of X.1

### Importing libraries

In [ ]:
from __future__ import annotations

from getpass import getpass
from pathlib import Path
import re

import pandas as pd
from ipywidgets import Dropdown, HTML
from pybis import Openbis

### Login and personalization

In [ ]:
# Enter your username, e.g. 'MMUSTER' or 'mmuster'
username = ''

Choose instance in drop-down menu.

In [ ]:
instances = ["main", "training", "playground", "demo"]

dd = Dropdown(options=instances, description="Select an instance:")
display(dd)

In [ ]:
instance = dd.value
o = Openbis(f"https://{instance}.datastore.bam.de")
print(f"Choosen instance (URL): {o.url}")

Choose file in drop-down menu.

In [ ]:
files_root = Path.cwd()
file_options = [
    (str(path.relative_to(files_root)), path)
    for path in sorted(files_root.rglob("*"), key=lambda item: str(item).casefold())
    if path.is_file()
]

if file_options:
    file_dropdown = Dropdown(
        options=file_options,
        description="Select a file:",
        layout={"width": "max-content"},
    )
    display(file_dropdown)
else:
    display(HTML(f"No files found in {files_root}"))

Execute the cell and enter the **password** in the field that would appear.

In [ ]:
password = getpass('Enter PASSWORD: ')
o.login(username, password)

Definition of user Space, i.e. your **Lab Notebook**

In [ ]:
division = o.get_object(sample_ident= f'/BAM_GLOBAL/BAM_DATA/PERSONS/{username.upper()}').props('bam_oe').removeprefix('OE_')
my_space = o.get_space(code=fr"{division}_{username}")
print(f'Space of your Lab Notebook: {my_space.code}')

Defining Entity types

In [ ]:
# Collections
type_collection = 'COLLECTION'

# Objects
type_chemical = 'CHEMICAL'
type_sample = 'SAMPLE'
type_instrument = 'INSTRUMENT'

## Registering Entities

Projects, Collections, Objects (Object type: CHEMICAL, INSTRUMENT) in:

### 1. Division (FB) inventory

#### Register Projects

In [ ]:
def get_or_create_project(space: str, code: str):
    project_id = f"/{space}/{code}"
    try:
        project = o.get_project(projectId=project_id)
        print(f"Already exists: {project.identifier}")
        return project
    except ValueError:
        project = o.new_project(space=space, code=code)
        project.save()
        print(f"Registered: {project.identifier}")
        return project

project_samples    = get_or_create_project(f"{division}_MATERIALS",  "Samples_Synthesis")
project_chemicals  = get_or_create_project(f"{division}_MATERIALS",  "Chemicals_Synthesis")

#### Register Collections

In [ ]:
def get_or_create_collection(code: str, name: str, default_type, project_id: str):
    collection_id = f"{project_id}/{code}"
    try:
        collection = o.get_collection(code=collection_id)
        print(f"Collection already exists: Name: {collection.props.get('$name')} | {collection.identifier}")
        return collection
    except ValueError:
        collection = o.new_collection(
            code=code,
            type=type_collection,
            project=project_id,
            props={"$name": name, "$default_object_type": default_type},
        )
        collection.save()
        print(f"Registered: {collection.identifier}")
        return collection

collection_chemicals   = get_or_create_collection("COLLECTION_CHEMICALS",   "Chemicals",   type_chemical,   project_chemicals.identifier)
collection_samples     = get_or_create_collection("COLLECTION_SAMPLES",     "Samples",     type_sample,     project_samples.identifier)

#### Register Objects

Object type: CHEMICAL

In [ ]:
chemicals_data = [
    {
        '$name': 'Chemical 1',
        'manufacturer': 'BAM',
        'hazardous_substance': False,
        'bam_oe': 'OE_X.1',
        'bam_location_complete': 'AH_8_05_1_O1_201',
    },
    {
        '$name': 'Chemical 2',
        'manufacturer': 'BAM',
        'hazardous_substance': False,
        'bam_oe': 'OE_X.1',
        'bam_location_complete': 'AH_8_05_1_O1_201',
    },
        {
        '$name': 'Treatment Solution',
        'manufacturer': 'BAM',
        'hazardous_substance': False,
        'bam_oe': 'OE_X.1',
        'bam_location_complete': 'AH_8_05_1_O1_201',
    }
]

def _norm_name(v: str) -> str:
    return " ".join((v or "").split()).strip().casefold()

def build_index():
    index = {}
    for obj in o.get_objects(collection=collection_chemicals.identifier, type=type_chemical):
        name = _norm_name(obj.props.get("$name"))
        if name:
            index[name] = obj
    return index

def get_or_create_chemical(chemical: dict, index: dict):
    name_raw = chemical.get("$name") or chemical.get("name")
    name_key = _norm_name(name_raw)
    if not name_key:
        raise ValueError("Chemical name missing (expected chemical['$name'] or chemical['name']).")

    existing = index.get(name_key)
    if existing is not None:
        print(f"Already exists: {existing.identifier} | {existing.props.get('$name')}")
        return existing

    props = dict(chemical)
    props["$name"] = (name_raw or "").strip()

    obj = o.new_object(
        type=type_chemical,
        collection=collection_chemicals.identifier,
        props=props,  # autogenerated
    ) 
    obj.save()
    index[name_key] = obj
    print(f"Registered: {obj.identifier} | {props['$name']}")
    return obj

chemical_index = build_index()
chemicals = [get_or_create_chemical(chem, chemical_index) for chem in chemicals_data]

### 2. BAM inventory

#### Register Collection

In [ ]:
def get_or_create_collection(code: str, name: str, default_type, project_id: str):
    collection_id = f"{project_id}/{code}"
    try:
        collection = o.get_collection(code=collection_id)
    except Exception:
        collection = None

    if collection is None:
        collection = o.new_collection(
            code=code,
            type=type_collection,
            project=project_id,
            props={"$name": name, "$default_object_type": default_type},
        )
        collection.save()
        print(f"Registered: {collection.identifier}")
    else:
        print(f"Already exists: Name: {collection.props["$name"]} with identifier: {collection.identifier}")

    return collection

collection_bam_inst = get_or_create_collection("COLLECTION_INSTRUMENTS", "Instruments",type_instrument,f"/BAM_EQUIPMENT/{division}_EQUIPMENT_OPEN")

#### Register Objects

Object type: INSTRUMENT

In [ ]:
instruments_bam_data = [
    {
        '$name': 'Instrument Structure',
        'manufacturer': 'BAM',
        'bam_oe': 'OE_X.1',
        'bam_location_complete': 'AH_8_05_1_O1_201',
    },
    {
        '$name': 'Instrument Parameter 1',
        'manufacturer': 'BAM',
        'bam_oe': 'OE_X.1',
        'bam_location_complete': 'AH_8_05_1_O1_201',
    },
    {
        '$name': 'Instrument Parameter 2',
        'manufacturer': 'BAM',
        'bam_oe': 'OE_X.1',
        'bam_location_complete': 'AH_8_05_1_O1_201',
    }
]

def build_instrument_index():
    index = {}
    for obj in o.get_objects(collection=collection_bam_inst.identifier, type=type_instrument):
        name = _norm_name(obj.props.get("$name"))
        if name:
            index[name] = obj
    return index

def get_or_create_instrument(ins: dict, index: dict):
    name_raw = ins.get("$name") or ins.get("name")
    name_key = _norm_name(name_raw)
    if not name_key:
        raise ValueError("Instrument name missing (expected ins['$name'] or ins['name']).")

    existing = index.get(name_key)
    if existing is not None:
        print(f"Already exists: {existing.identifier} | {existing.props.get('$name')}")
        return existing

    props = dict(ins)
    props["$name"] = (name_raw or "").strip()

    obj = o.new_object(
        type=type_instrument,
        collection=collection_bam_inst.identifier,
        props=props,  # autogenerated
    ) 
    obj.save()
    index[name_key] = obj
    print(f"Registered: {obj.identifier} | {props['$name']}")
    return obj

instrument_index = build_instrument_index()
instruments = [get_or_create_instrument(ins, instrument_index) for ins in instruments_bam_data]